In [2]:
pip install pandas numpy scikit-learn imblearn lightgbm xgboost catboost shap lime matplotlib seaborn joblib openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 70.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.6.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U scikit-learn imbalanced-learn

  Using cached scikit_learn-1.7.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 48.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 3.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: imbalanced-learn
    Found existing installation: imbalanced-learn 0.13.0
    Uninstalling imbalanced-learn-0.13.0:
      Successfully uninstalled imbalanced-learn-0.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have nu

In [4]:
!pip install scikit-survival --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 33.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.3/298.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.1/220.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00:00:01


In [3]:
!pip install lifelines


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 4.7 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=883b512f1300eca38309c95b677d06e0463bffec88e57a150be32803ee069ba9
  Stored in directory: /root/.cache/pip/wheels/8b/67/f4/2caaae2146198dcb824f31a303833b07b14a5ec863fb3acd7b
Successfully built autograd-gamma


Step 1: Imports & Setup

In [3]:
# Basic
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.utils import resample

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Boosting libs
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Survival analysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest

# Calibration
from sklearn.calibration import CalibratedClassifierCV

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve
)

# Plots
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

# Explainability
import shap
import lime.lime_tabular

# Misc
import joblib
import warnings
warnings.filterwarnings("ignore")


Step 2: Load Datasets

In [ ]:
# Replace with your local paths
kidney_df = pd.read_csv("E:\\Journal\\CKD\\kidney_disease.csv")
clinical_df = pd.read_excel("E:\\Journal\\CKD\\pone.0199920.s002.xlsx")

print("Kidney dataset shape:", kidney_df.shape)
print("Clinical dataset shape:", clinical_df.shape)


Kidney dataset shape: (400, 26)
Clinical dataset shape: (491, 25)


Step 3: Preprocessing – Kidney Dataset (Detection)

In [6]:
# Clean target
kidney_df["classification"] = kidney_df["classification"].str.strip().map({"ckd": 1, "notckd": 0})

# Replace "?" with NaN
kidney_df = kidney_df.replace("?", np.nan)

# Convert numeric-like cols
num_cols = ['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']
for col in num_cols:
    kidney_df[col] = pd.to_numeric(kidney_df[col], errors="coerce")

# Clinically aware imputation
kidney_df["sod"] = kidney_df["sod"].fillna(140)  # sodium normal mean
kidney_df["pot"] = kidney_df["pot"].fillna(4.5)  # potassium normal mean
kidney_df["hemo"] = kidney_df["hemo"].fillna(13.5)  # hemoglobin normal mean

for col in ["pcv","wc","rc"]:
    kidney_df[col] = kidney_df[col].fillna(kidney_df[col].median())

# Feature engineering
kidney_df["urea_creatinine_ratio"] = kidney_df["bu"] / (kidney_df["sc"]+1e-5)
kidney_df["anemia_flag"] = (kidney_df["hemo"] < 12).astype(int)
kidney_df["htn_dm_interaction"] = ((kidney_df["htn"]=="yes") & (kidney_df["dm"]=="yes")).astype(int)

# Outlier handling (winsorize)
kidney_df["sc"] = np.clip(kidney_df["sc"], 0, 15)  # serum creatinine
kidney_df["bu"] = np.clip(kidney_df["bu"], 0, 300) # blood urea
kidney_df["bp"] = np.clip(kidney_df["bp"], 40, 200)

# Split features / target
X_kd = kidney_df.drop(columns=["id","classification"])
y_kd = kidney_df["classification"]

print("Kidney dataset ready. Features:", X_kd.shape[1])


Kidney dataset ready. Features: 27


Step 4: Preprocessing – Clinical Dataset (Prognosis)

In [7]:
# Clean column names
clinical_df = clinical_df.rename(columns={"HistoryHTN ": "HistoryHTN"})

# -----------------------------
# Target (Survival outcome)
# -----------------------------
# Structured survival outcome (event, time)
y_clinical = np.array([
    (bool(event), time) for event, time in zip(clinical_df["EventCKD35"], clinical_df["TimeToEventMonths"])
], dtype=[("event", "bool"), ("time", "int")])

# -----------------------------
# Feature Engineering
# -----------------------------
clinical_df["BMI_Category"] = pd.cut(
    clinical_df["BMIBaseline"],
    bins=[0, 25, 30, 100],
    labels=["Normal", "Overweight", "Obese"]
)

clinical_df["Metabolic_Syndrome"] = (
    clinical_df["HistoryDiabetes"] +
    clinical_df["HistoryHTN"] +
    clinical_df["HistoryObesity"] +
    clinical_df["HistoryDLD"]
)

# Age group dummies (align with Age.3.categories)
clinical_df = pd.get_dummies(clinical_df, columns=["Age.3.categories"], drop_first=True)

# -----------------------------
# Missing Value Imputation
# -----------------------------
# HbA1C & Triglycerides baseline: median imputation
clinical_df["HgbA1C"] = clinical_df["HgbA1C"].fillna(clinical_df["HgbA1C"].median())
clinical_df["TriglyceridesBaseline"] = clinical_df["TriglyceridesBaseline"].fillna(
    clinical_df["TriglyceridesBaseline"].median()
)

# -----------------------------
# Outlier Handling (Winsorize)
# -----------------------------
clinical_df["CreatnineBaseline"] = np.clip(clinical_df["CreatnineBaseline"], 20, 800)
clinical_df["CholesterolBaseline"] = np.clip(clinical_df["CholesterolBaseline"], 2, 15)
clinical_df["TriglyceridesBaseline"] = np.clip(clinical_df["TriglyceridesBaseline"], 0.2, 10)

# -----------------------------
# Feature / Target Split
# -----------------------------
X_clinical = clinical_df.drop(columns=["StudyID", "EventCKD35", "TimeToEventMonths"])

print("Clinical dataset ready. Features:", X_clinical.shape[1])
print("Survival outcome shape:", y_clinical.shape)


Clinical dataset ready. Features: 25
Survival outcome shape: (491,)


Step 5: Detection Models – Kidney Dataset

In [ ]:
# -----------------------------
# Train-Test Split
# -----------------------------
X_train_kd, X_test_kd, y_train_kd, y_test_kd = train_test_split(
    X_kd, y_kd, test_size=0.2, stratify=y_kd, random_state=42
)

# -----------------------------
# Preprocessing Pipeline for Kidney Dataset
# -----------------------------
# Identify numeric and categorical columns
numeric_features_kd = X_kd.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_kd = X_kd.select_dtypes(include=['object', 'category']).columns.tolist()

# Create preprocessing pipeline with imputation
preprocessor_kd = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features_kd),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features_kd)
    ]
)

# -----------------------------
# Preprocess numeric/categorical
# -----------------------------
X_train_proc = preprocessor_kd.fit_transform(X_train_kd)
X_test_proc = preprocessor_kd.transform(X_test_kd)

# Check for any remaining NaN values
print(f"NaN values in X_train_proc: {np.isnan(X_train_proc).sum()}")
print(f"NaN values in X_test_proc: {np.isnan(X_test_proc).sum()}")

# -----------------------------
# Apply SMOTE once
# -----------------------------
smote = BorderlineSMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train_kd)

print(f"✅ SMOTE applied successfully")
print(f"✅ Resampled training set shape: {X_train_res.shape}")
print(f"✅ Resampled target distribution: {np.bincount(y_train_res)}")

# -----------------------------
# Define Base Classifiers
# -----------------------------
base_classifiers = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "CatBoost": cb.CatBoostClassifier(random_state=42, verbose=False),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "MLP": MLPClassifier(random_state=42),
    "Balanced RF": BalancedRandomForestClassifier(random_state=42),
    "Easy Ensemble": EasyEnsembleClassifier(random_state=42)
}

# -----------------------------
# Cross-Validation Evaluation
# -----------------------------
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline as SklearnPipeline

# Create CV pipeline with preprocessing + SMOTE + classifier
cv_results = {}
cv_scores = {}

print("🔄 Running 5-Fold Stratified Cross-Validation...")
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in base_classifiers.items():
    # Create pipeline: preprocessor -> SMOTE -> classifier
    cv_pipeline = SklearnPipeline([
        ('preprocessor', preprocessor_kd),
        ('smote', BorderlineSMOTE(random_state=42)),
        ('classifier', clf)
    ])
    
    # Cross-validation scores
    cv_auc = cross_val_score(cv_pipeline, X_kd, y_kd, cv=cv_folds, scoring='roc_auc')
    cv_f1 = cross_val_score(cv_pipeline, X_kd, y_kd, cv=cv_folds, scoring='f1')
    cv_acc = cross_val_score(cv_pipeline, X_kd, y_kd, cv=cv_folds, scoring='accuracy')
    
    cv_results[name] = {
        'AUC_mean': cv_auc.mean(),
        'AUC_std': cv_auc.std(),
        'F1_mean': cv_f1.mean(),
        'F1_std': cv_f1.std(),
        'Accuracy_mean': cv_acc.mean(),
        'Accuracy_std': cv_acc.std()
    }
    
    cv_scores[name] = cv_auc.mean()

print("✅ Cross-validation completed!")

# -----------------------------
# Train individual classifiers (no ensembles)
# -----------------------------
results_detection = []
fitted_classifiers = {}

for name, clf in base_classifiers.items():
    clf.fit(X_train_res, y_train_res)
    fitted_classifiers[name] = clf
    
    y_pred = clf.predict(X_test_proc)
    y_prob = clf.predict_proba(X_test_proc)[:,1]
    
    acc = accuracy_score(y_test_kd, y_pred)
    f1 = f1_score(y_test_kd, y_pred)
    prec = precision_score(y_test_kd, y_pred)
    rec = recall_score(y_test_kd, y_pred)
    auc_score = roc_auc_score(y_test_kd, y_prob)
    pr_auc = average_precision_score(y_test_kd, y_prob)
    brier = brier_score_loss(y_test_kd, y_prob)
    
    # Add CV scores
    cv_auc_mean = cv_results[name]['AUC_mean']
    cv_auc_std = cv_results[name]['AUC_std']
    
    results_detection.append([name, acc, f1, prec, rec, auc_score, pr_auc, brier, cv_auc_mean, cv_auc_std])

results_detection_df = pd.DataFrame(
    results_detection,
    columns=["Model","Accuracy","F1","Precision","Recall","ROC AUC","PR AUC","Brier Score","CV AUC Mean","CV AUC Std"]
).sort_values(by="CV AUC Mean", ascending=False)

print("\n=== Kidney Dataset Detection Results (with Cross-Validation) ===")
print(results_detection_df)

# -----------------------------
# Select Best Model and Calibrate
# -----------------------------
best_model_name = results_detection_df.iloc[0]["Model"]
best_model = fitted_classifiers[best_model_name]

# Calibrate the best model
calibrated_clf = CalibratedClassifierCV(best_model, method='isotonic', cv=3)
calibrated_clf.fit(X_train_res, y_train_res)

print(f"\n✅ Best Detection Model: {best_model_name}")
print(f"✅ CV AUC Score: {cv_scores[best_model_name]:.3f} ± {cv_results[best_model_name]['AUC_std']:.3f}")
print("✅ Model calibrated for better probability estimates")


[LightGBM] [Info] Number of positive: 200, number of negative: 120
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000287 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 694
[LightGBM] [Info] Number of data points in the train set: 320, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.625000 -> initscore=0.510826
[LightGBM] [Info] Start training from score 0.510826
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

Step 6: Clinical Dataset Prognosis Models

In [ ]:
# -----------------------------
# Clinical Dataset Train-Test Split
# -----------------------------
X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(
    X_clinical, y_clinical, test_size=0.2, random_state=42
)

# -----------------------------
# Preprocessing Pipeline for Clinical Dataset
# -----------------------------
# Identify numeric and categorical columns
numeric_features_cl = X_clinical.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_cl = X_clinical.select_dtypes(include=['object', 'category']).columns.tolist()

# Create preprocessing pipeline with imputation
preprocessor_cl = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features_cl),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features_cl)
    ]
)

# Check for any remaining NaN values after preprocessing
X_train_cl_proc = preprocessor_cl.fit_transform(X_train_cl)
X_test_cl_proc = preprocessor_cl.transform(X_test_cl)

print(f"NaN values in X_train_cl_proc: {np.isnan(X_train_cl_proc).sum()}")
print(f"NaN values in X_test_cl_proc: {np.isnan(X_test_cl_proc).sum()}")

# -----------------------------
# Survival Models
# -----------------------------
# CoxPH
cox_pipe = Pipeline([
    ("preprocessor", preprocessor_cl),
    ("model", CoxPHSurvivalAnalysis())
])
cox_pipe.fit(X_train_cl, y_train_cl)

# RSF
rsf_pipe = Pipeline([
    ("preprocessor", preprocessor_cl),
    ("model", RandomSurvivalForest(n_estimators=200, random_state=42))
])
rsf_pipe.fit(X_train_cl, y_train_cl)

# Store models
prognosis_models = {
    "Cox Proportional Hazards": cox_pipe,
    "Random Survival Forest": rsf_pipe
}

# Pick RSF as best
prognosis_model = rsf_pipe
print("✅ Prognosis model = Random Survival Forest (with OneHot categorical handling)")
print("✅ Cox Proportional Hazards trained successfully")
print("✅ Random Survival Forest trained successfully")


ValueError: Cannot cast object dtype to float64

Step 7: Hybrid Decision Function

In [ ]:
def hybrid_ckd_predict(patient_kd, patient_cl, detection_model, prognosis_model, horizon=60):
    """
    Hybrid CKD Prediction Function
    
    Combines detection (current CKD status) and prognosis (future CKD risk) models
    to provide comprehensive clinical decision support for CKD management.
    
    Parameters
    ----------
    patient_kd : pandas.DataFrame
        Single patient data from kidney dataset with shape (1, n_features).
        Contains clinical features for CKD detection (age, creatinine, etc.)
    
    patient_cl : pandas.DataFrame  
        Single patient data from clinical dataset with shape (1, n_features).
        Contains features for CKD prognosis (BMI, comorbidities, etc.)
    
    detection_model : sklearn.CalibratedClassifierCV
        Trained and calibrated classification model for CKD detection.
        Should have .predict() and .predict_proba() methods.
    
    prognosis_model : sklearn.pipeline.Pipeline
        Trained survival analysis model for CKD prognosis.
        Should have .predict_survival_function() method.
    
    horizon : int, default=60
        Prediction horizon in months for prognosis model.
        Common values: 12, 24, 36, 60 months.
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'Detection': str
            "CKD Present" or "CKD Not Present"
        - 'Detection Probability': float
            Probability of current CKD (0-1)
        - 'Future Risk (X months)': float
            Risk of developing CKD in future (0-1)
    
    Notes
    -----
    Clinical Logic:
    1. If patient already has CKD (detection=1), future risk = 100%
    2. If patient doesn't have CKD (detection=0), use prognosis model
    3. Risk = 1 - Survival Probability at given horizon
    
    Examples
    --------
    >>> result = hybrid_ckd_predict(
    ...     sample_kd, sample_cl, 
    ...     calibrated_clf, prognosis_model,
    ...     horizon=60
    ... )
    >>> print(result['Detection'])
    'CKD Not Present'
    >>> print(result['Future Risk (60 months)'])
    0.234
    """
    # Detection
    det_pred = detection_model.predict(patient_kd)[0]
    det_prob = detection_model.predict_proba(patient_kd)[0,1]
    
    if det_pred == 1:  # CKD present
        return {
            "Detection": "CKD Present",
            "Detection Probability": round(det_prob, 3),
            "Future Risk ({} months)".format(horizon): 1.0  # already CKD
        }
    else:  # CKD not present → run prognosis
        surv_fn = prognosis_model.predict_survival_function(patient_cl)[0]
        surv_prob = surv_fn(horizon)
        risk_prob = 1 - surv_prob  # risk = 1 - survival
        return {
            "Detection": "CKD Not Present",
            "Detection Probability": round(det_prob, 3),
            "Future Risk ({} months)".format(horizon): round(risk_prob, 3)
        }

# -----------------------------
# Demo on first patient
# -----------------------------
sample_kd = X_test_kd.iloc[[0]]
sample_cl = X_test_cl.iloc[[0]]

hybrid_result = hybrid_ckd_predict(
    sample_kd, sample_cl,
    calibrated_clf,  # best calibrated detection model
    prognosis_models["Random Survival Forest"], # trained RSF
    horizon=60
)

print("\n=== Hybrid Decision for Sample Patient ===")
print(hybrid_result)


Step 8A: Evaluation Improvements (Detection Model)

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve

# -----------------------------
# Calibration Curve (Kidney dataset)
# -----------------------------
y_prob_det = calibrated_clf.predict_proba(X_test_kd)[:,1]
prob_true, prob_pred = calibration_curve(y_test_kd, y_prob_det, n_bins=10)

plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker='o', label='Detection Model')
plt.plot([0,1],[0,1],'k--', label='Perfect Calibration')
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve (Kidney Detection Model)")
plt.legend()
plt.show()

# -----------------------------
# Precision-Recall Curve (Imbalance check)
# -----------------------------
prec, rec, thr = precision_recall_curve(y_test_kd, y_prob_det)
plt.figure(figsize=(6,6))
plt.plot(rec, prec, label='PR Curve (AUC = %.3f)' % average_precision_score(y_test_kd, y_prob_det))
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()


Step 8B: Decision Curve Analysis (Clinical Prognosis Model)

In [ ]:
# Simple Decision Curve Approximation
times = [12, 24, 36, 60]  # months
surv_funcs = prognosis_models["Random Survival Forest"].predict_survival_function(X_test_cl)

plt.figure(figsize=(8,6))
for i, t in enumerate(times):
    risks = [1 - fn(t) for fn in surv_funcs]  # risk = 1-survival
    plt.hist(risks, bins=20, alpha=0.5, label=f"Risk at {t} months")

plt.xlabel("Predicted Risk Probability")
plt.ylabel("Patient Count")
plt.title("Decision Distribution (Clinical Prognosis Model)")
plt.legend()
plt.show()


Step 8C: Explainability – SHAP

In [ ]:
# -----------------------------
# SHAP for Detection Model
# -----------------------------
explainer_det = shap.Explainer(calibrated_clf, X_train_kd)
shap_values_det = explainer_det(X_test_kd[:50])

# Global summary
shap.summary_plot(shap_values_det, X_test_kd[:50], plot_type="bar")

# Dependence plot (Age vs Creatinine)
if "age" in X_test_kd.columns and "sc" in X_test_kd.columns:
    shap.dependence_plot("sc", shap_values_det.values, X_test_kd[:50], interaction_index="age")


Step 8D: Explainability – LIME

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train_kd),
    feature_names=X_train_kd.columns,
    class_names=["Not CKD","CKD"],
    mode="classification"
)

exp = lime_explainer.explain_instance(
    data_row=X_test_kd.iloc[0],
    predict_fn=calibrated_clf.predict_proba
)

exp.show_in_notebook(show_table=True)


Step 8E: Global Surrogate Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

# Surrogate decision tree trained on ensemble predictions
y_surrogate = calibrated_clf.predict(X_train_kd)

surrogate = DecisionTreeClassifier(max_depth=3, random_state=42)
surrogate.fit(X_train_kd, y_surrogate)

plt.figure(figsize=(12,8))
tree.plot_tree(surrogate, feature_names=X_train_kd.columns, class_names=["Not CKD","CKD"], filled=True)
plt.title("Global Surrogate Model for Detection Ensemble")
plt.show()


🟢 Preprocessing Improvements (Detection + Prognosis Models এর জন্য)
1. Missing Value Imputation (Clinically aware)

Sodium (Na) → normal range (135–145 mmol/L) এর mean দিয়ে fill করুন।

Potassium (K) → normal range (3.5–5.0 mmol/L)।

Hemoglobin (Hb) → normal range (12–16 g/dL)।

Blood cell counts (PCV, WC, RC) → median অথবা regression imputation।

Better alternative → IterativeImputer অথবা KNNImputer।

2. Feature Engineering
Kidney dataset:

Urea-to-Creatinine Ratio → bu / sc

Anemia flag → 1 if hemo < 12 else 0

Hypertension + Diabetes interaction → htn * dm

Clinical dataset:

BMI Category → (Normal, Overweight, Obese)

Metabolic Syndrome Score → combine (HTN, DM, Obesity, Dyslipidemia)

Age group dummies → align with Age.3.categories

3. Outlier Handling

Creatinine, Urea, BP extreme values → remove না করে clinically impossible values clip/winsorize করুন।
উদাহরণ: creatinine > 15 mg/dL unlikely → cap at 15।

4. Class Imbalance

Kidney dataset → BorderlineSMOTE (already good)

Clinical dataset → outcome rare হলে Stratified sampling + oversampling mix করুন।

5. Survival Data Preparation (Clinical dataset)

Outcome শুধু binary না, TimeToEventMonths numeric রাখুন।

Models: CoxPH, RSF → input = (X, y_time, y_event)।

6. Scaling + Encoding

StandardScaler → age, BP, hemoglobin ইত্যাদি continuous numeric features।

RobustScaler → skewed features (Creatinine, Urea, Triglycerides)।

Binary categorical → OrdinalEncoder।

Non-binary categorical → OneHotEncoder(handle_unknown="ignore") (scikit-survival numeric required)।

7. Probability Calibration

Detection model (kidney) → calibrated probability (Platt / Isotonic Regression)।

Prognosis model (clinical) → survival probability curve বা risk score।

🟢 Optional / Nice-to-Have (না দিলেও হবে)

Multicollinearity check (VIF, PCA) → optional।

Rare category handling → optional (যদি খুব imbalance না থাকে)।

Skewed variable log-transform → optional।

Time-dependent covariates → advanced, overkill for paper।

✅ এই preprocessing improvements গুলো detection + prognosis দুই মডেলের জন্য solid clinical justification সহ।
Reviewer দেখবে: “data cleaning + feature engineering clinically grounded, not blind ML only”।

👉 ভাই, চান আমি এগুলোকে একটা ready-to-use preprocessing function আকারে লিখে দিই (Kidney + Clinical dataset আলাদা config সহ)?

You said:
model?
ChatGPT said:

আচ্ছা ভাই, এখন আমি আপনার জন্য Model Improvements Summary লিখে দিচ্ছি – যেগুলো detection (diagnosis) আর prognosis (prediction) model-এ add করলে research আরও শক্ত হবে।

🟢 Model Improvements (Detection + Prognosis)
🔹 1. Detection Model (Kidney Dataset → আজ CKD আছে কি না?)

Already used:

Logistic Regression, RandomForest, XGBoost, LightGBM, CatBoost, Gradient Boosting

Naive Bayes, KNN, Decision Tree, MLP, BalancedRF, EasyEnsemble

Improvements:

Use BorderlineSMOTE → good for imbalanced data.

CalibratedClassifierCV → probability calibration (Platt scaling / Isotonic regression)। এতে output probability (risk %) ডাক্তারদের জন্য clinically useful হবে।

Report extra metrics: PR-AUC, Brier Score, Calibration Curve।

Best-performing (likely) → RandomForest / ExtraTrees / GradientBoosting (clinically robust + explainable)।

🔹 2. Prognosis Model (Clinical Dataset → ভবিষ্যতে CKD risk কত?)

Options:

(a) Classical Survival Models

Cox Proportional Hazards (CoxPH) → gold standard baseline।

Random Survival Forest (RSF) → non-linear, handles interactions.

(b) Advanced Survival Models

DeepSurv (neural net Cox) → non-linear deep version.

DeepHit (competing risk) → more complex but publishable.

Best trade-off for paper → CoxPH (easy interpretability) + RSF (better accuracy)।

🔹 3. Hybrid Clinical Decision Pipeline

Patient → Detection model (আজ CKD আছে?)

If CKD present → prognosis = 100% (already diseased)।

If CKD absent → prognosis model predict করবে 5-year (60 month) risk probability।

👉 এই approach diagnosis + prognosis দুইটাই একসাথে দেবে → novelty + clinical value।

🔹 4. Evaluation Metrics

Detection model: Accuracy, F1, ROC-AUC, PR-AUC, Brier Score, Calibration curve।

Prognosis model: Concordance index (C-index), Brier score (time-dependent), Survival curve plots।

Hybrid: Decision Curve Analysis (DCA) → clinical utility measure।

🔹 5. Explainability

SHAP summary plots → feature global importance।

SHAP dependence plots → interaction (e.g. Creatinine × Age)।

LIME local explanations → per-patient explainability।

Global surrogate tree → ensemble model summarised as decision tree → easy for clinicians।

✅ Bottom Line

Detection → ensemble tree-based (RandomForest/LightGBM) + calibrated probability।

Prognosis → CoxPH + RSF combo → interpretability + accuracy।

Hybrid → integrates diagnosis (today) + prognosis (future)।

Evaluation → add PR-AUC, Brier, Calibration, C-index, DCA।

Explainability → SHAP, LIME, Surrogate tree।

🔹 1. Preprocessing Improvements
✅ Missing Value Imputation (Clinically Aware)

Sodium (Na) → normal range (135–145 mmol/L) এর mean দিয়ে fill করুন।

Potassium (K) → normal range (3.5–5.0 mmol/L)।

Hemoglobin (Hb) → normal range (12–16 g/dL)।

Blood counts (PCV, WC, RC) → median বা regression imputation।

Advanced → KNNImputer বা IterativeImputer।

👉 Clinical justification: missing মানকে “statistical average” না, বরং physiological normal range অনুযায়ী handle করা → realistic patient representation।

✅ Feature Engineering

Kidney dataset (Detection):

Urea-to-Creatinine Ratio → bu / sc (renal function marker)।

Anemia flag → 1 if hemo < 12 else 0।

HTN × DM interaction → diabetes+hypertension একসাথে থাকলে CKD risk boost করে।

Clinical dataset (Prognosis):

BMI Category → WHO cut-off অনুযায়ী (Normal, Overweight, Obese)।

Metabolic Syndrome Score → sum of (HTN, DM, Obesity, Dyslipidemia)।

Age group dummies → align with Age.3.categories (young, middle, old)।

👉 Clinical justification: Derived variables physicians actually use in practice (anemia, metabolic syndrome, etc.)।

✅ Outlier Handling

Creatinine, Urea, BP extreme values winsorize করুন (clinically impossible cut-offs)।
e.g. Creatinine > 15 mg/dL → cap at 15।

Outlier delete করবেন না (small dataset)।

✅ Class Imbalance

Kidney dataset → BorderlineSMOTE (rare minority cases balance)।

Clinical dataset → Stratified sampling + oversampling (event rare)।

👉 Clinical justification: ensures both CKD-positive & CKD-negative properly represented।

✅ Scaling + Encoding

StandardScaler → age, BP, hemoglobin।

RobustScaler → skewed features (Creatinine, Urea, Triglycerides)।

Binary categorical → OrdinalEncoder।

Non-binary categorical → OneHotEncoder(handle_unknown="ignore") → survival models require numeric।

✅ Survival Data Preparation

Outcome শুধু binary না, time-to-event (months) রাখুন।

Final dataset: (features, y_time, y_event) → CoxPH / RSF input।

🔹 2. Detection Model (Kidney Dataset)
✅ Baseline Classifiers

Already tried: Logistic Regression, RandomForest, XGBoost, LightGBM, CatBoost, Gradient Boosting, Naive Bayes, KNN, Extra Trees, Decision Tree, MLP, Balanced RF, Easy Ensemble।

✅ Improvements

Probability calibration → Platt scaling / Isotonic regression।
→ Output risk % clinically interpretable।

Balanced ensembles → BalancedRandomForestClassifier, EasyEnsembleClassifier।

Evaluation beyond accuracy →

PR-AUC (class imbalance sensitive)

Brier Score (probabilistic prediction quality)

Calibration Curve

👉 Best candidates: RandomForest / ExtraTrees / LightGBM (stable, high AUC, interpretable)।

🔹 3. Prognosis Model (Clinical Dataset)
✅ Classical Survival Models

Cox Proportional Hazards (CoxPH) → interpretable hazard ratios।

Random Survival Forest (RSF) → non-linear, handles interactions।

✅ Advanced Survival Models

DeepSurv → neural network CoxPH (non-linear risk prediction)।

DeepHit → multi-state outcome (future upgrade)।

✅ Evaluation Metrics

Concordance index (C-index) → discrimination ability।

Brier Score (time-dependent) → calibration quality।

Kaplan-Meier survival curves → group-wise visualization।

👉 Clinical justification: CoxPH → explainable (HR > 1 = risk ↑)। RSF → accuracy।

🔹 4. Hybrid Clinical Pipeline
Patient
   ↓
Detection Model (Kidney dataset)
   → If CKD = Present → Prognosis = 100%
   → If CKD = Absent → Prognosis model → Future risk %


👉 Value:

একসাথে diagnosis (আজ CKD?) + prognosis (ভবিষ্যতে CKD risk %)

Novelty: Combined decision support tool → clinician-friendly।

🔹 5. Evaluation Improvements
✅ Detection (Kidney)

Accuracy, F1, ROC-AUC

Precision-Recall AUC

Brier Score

Calibration Curve

✅ Prognosis (Clinical)

C-index

Time-dependent Brier Score

Decision Curve Analysis (DCA) → clinical utility

🔹 6. Explainability

SHAP summary plots → global feature importance।

SHAP dependence plots → interaction (Age × Creatinine)।

LIME local explanations → per-patient risk breakdown।

Global surrogate tree → black-box ensemble summarised in decision tree।

👉 Clinical justification: Doctors will trust model outputs only if transparent।

✅ Final Takeaway

Detection model → Calibrated ensemble (RF/LightGBM) with SMOTE.

Prognosis model → RSF + CoxPH survival analysis.

Hybrid pipeline → “Today diagnosis + Future prediction”.

Evaluation → Go beyond accuracy (PR-AUC, Brier, C-index, DCA).

Explainability → SHAP + LIME + Surrogate model.

👉 ভাই, আপনি চাইলে আমি এগুলোকে একটা ready-to-use model architecture diagram (flowchart figure) আকারে বানিয়ে দিতে পারি, যেটা সরাসরি আপনার paper এ বসানো যাবে। দেব?